# 07 : Plotly

강의자료: `docs/course/07_Plotly.md`

Plotly Express로 인터랙티브 그래프를 만들고 HTML로 저장한다.
Matplotlib과 달리 마우스를 올리면 값이 보이고, 확대·필터가 된다.

## 데이터

- `px.data.gapminder()` — Plotly 내장. 1952~2007년 142개국 기대수명·인구·1인당 GDP
- `data/raw/seoul_rent_2026.csv` — 과제 2용 (05에서 쓴 파일)

저장 위치는 `outputs/html/`.

> 강의자료 학습목표에 서울 전월세 전처리가 있으나 본문 코드는 Gapminder만 다룬다.
> 전처리 부분은 실습 저장소 `03_2_plotly.ipynb`를 참고해 과제 2에 구현했다.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

HTML = Path("outputs/html")
HTML.mkdir(parents=True, exist_ok=True)

gap = px.data.gapminder()
print(gap.shape, list(gap.columns))
gap.head()

(1704, 8) ['country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap', 'iso_alpha', 'iso_num']


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


## 1. 산점도 — 기본에서 디자인까지

In [2]:
g07 = gap.query("year == 2007")
print("2007년 국가 수:", len(g07))

fig = px.scatter(g07, x="gdpPercap", y="lifeExp")
fig.write_html(HTML / "1_1.html", include_plotlyjs="cdn")
fig.show()

2007년 국가 수: 142


In [3]:
fig = px.scatter(
    g07, x="gdpPercap", y="lifeExp",
    size="pop", color="continent", hover_name="country",
    log_x=True,                 # GDP는 편차가 커서 로그 축이 읽기 쉽다
    size_max=55,
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={"gdpPercap": "1인당 GDP (달러, 로그)", "lifeExp": "기대수명 (년)",
            "continent": "대륙", "pop": "인구"},
)
fig.update_layout(title="1인당 GDP와 기대수명 (2007)",
                  template="plotly_white", height=520)
fig.update_xaxes(showgrid=True, gridcolor="#eee")
fig.update_yaxes(showgrid=True, gridcolor="#eee")
fig.write_html(HTML / "1_2.html", include_plotlyjs="cdn")
fig.show()

## 2. Facet — 연도별로 쪼개 보기

In [4]:
fig = px.scatter(
    gap.query("year >= 1982"), x="gdpPercap", y="lifeExp",
    size="pop", color="continent", hover_name="country",
    log_x=True, size_max=35,
    facet_col="year", facet_col_wrap=3,
    labels={"gdpPercap": "1인당 GDP", "lifeExp": "기대수명"},
)
fig.update_xaxes(matches="x")   # 패널 간 축을 맞춰야 비교가 된다
fig.update_yaxes(matches="y")
fig.update_layout(height=620, template="plotly_white")
fig.write_html(HTML / "1_3.html", include_plotlyjs="cdn")
fig.show()

## 3. 애니메이션 — 시간 흐름

In [5]:
fig = px.scatter(
    gap, x="gdpPercap", y="lifeExp",
    size="pop", color="continent", hover_name="country",
    animation_frame="year", animation_group="country",
    log_x=True, size_max=55,
    range_x=[200, 100000], range_y=[25, 90],   # 축을 고정해야 프레임이 튀지 않는다
    labels={"gdpPercap": "1인당 GDP", "lifeExp": "기대수명"},
)
fig.update_layout(title="1952-2007 변화", template="plotly_white", height=560)
fig.write_html(HTML / "1_4.html", include_plotlyjs="cdn")
fig.show()

## 4. Bar

In [6]:
asia07 = gap.query("year == 2007 and continent == 'Asia'").nlargest(15, "pop")

fig = px.bar(asia07.sort_values("pop"), x="pop", y="country",
             orientation="h", color="lifeExp",
             color_continuous_scale="Blues",
             labels={"pop": "인구", "country": "", "lifeExp": "기대수명"})
fig.update_layout(title="아시아 인구 상위 15개국 (2007)",
                  template="plotly_white", height=520)
fig.write_html(HTML / "2_3.html", include_plotlyjs="cdn")
fig.show()

## 5. Histogram · Box

In [7]:
fig = px.histogram(g07, x="lifeExp", nbins=30,
                   histnorm="probability density",
                   labels={"lifeExp": "기대수명"})
fig.update_layout(title="기대수명 분포 (2007)", template="plotly_white", height=380)
fig.write_html(HTML / "3_2.html", include_plotlyjs="cdn")
fig.show()

fig = px.box(g07, x="continent", y="lifeExp", color="continent",
             labels={"continent": "대륙", "lifeExp": "기대수명"})
fig.update_layout(title="대륙별 기대수명 (2007)", template="plotly_white",
                  height=420, showlegend=False)
fig.write_html(HTML / "4_2.html", include_plotlyjs="cdn")
fig.show()

## 6. Line · Area — 추세

In [8]:
asia = gap[gap["continent"] == "Asia"]
top6 = asia.query("year == 2007").nlargest(6, "pop")["country"]

fig = px.line(asia[asia["country"].isin(top6)],
              x="year", y="lifeExp", color="country", markers=True,
              labels={"year": "연도", "lifeExp": "기대수명", "country": "국가"})
fig.update_layout(title="아시아 주요국 기대수명 추이", template="plotly_white", height=440)
fig.write_html(HTML / "5_1.html", include_plotlyjs="cdn")
fig.show()

fig = px.area(asia.groupby(["year", "country"], as_index=False)["pop"].sum(),
              x="year", y="pop", color="country",
              labels={"year": "연도", "pop": "인구"})
fig.update_layout(title="아시아 인구 누적", template="plotly_white",
                  height=440, showlegend=False)
fig.write_html(HTML / "6_1.html", include_plotlyjs="cdn")
fig.show()

## 과제 2 — 서울 전세 가격 분석

월세를 전세로 환산해 비교 가능한 지표를 만들고, 면적으로 나눠 단위 가격을 본다.

**전월세 전환율 5%** 가정: 환산보증금 = 보증금 + (월세 x 12 / 0.05)

In [9]:
CODE_COLS = {"CGG_CD": "string", "STDG_CD": "string"}
rent = pd.read_csv("data/raw/seoul_rent_2026.csv", encoding="utf-8-sig", dtype=CODE_COLS)
print(rent.shape)

df = rent[["CGG_NM", "STDG_NM", "BLDG_USG", "RENT_SE",
           "GRFE", "RTFE", "RENT_AREA", "CTRT_DAY", "FLR"]].copy()

# 전월세 전환율 5% — 월세를 전세 보증금으로 환산해 같은 잣대로 비교한다
CONVERSION_RATE = 0.05
df["JEONSE_CONVERTED"] = df["GRFE"] + (df["RTFE"] * 12 / CONVERSION_RATE)

# 면적으로 나눠 단위 가격을 만든다. 면적이 0이면 나눗셈이 깨지므로 먼저 거른다
df = df[df["RENT_AREA"] > 0]
df["JEONSE_PER_M2"] = (df["JEONSE_CONVERTED"] / df["RENT_AREA"]).round(2)

print(df[["RENT_SE", "GRFE", "RTFE", "JEONSE_CONVERTED", "RENT_AREA", "JEONSE_PER_M2"]].head())

(416037, 23)
  RENT_SE   GRFE  RTFE  JEONSE_CONVERTED  RENT_AREA  JEONSE_PER_M2
0      전세   8085     0            8085.0      12.05         670.95
1      월세   8266    13           11386.0      59.99         189.80
2      월세   3000   195           49800.0      42.17        1180.93
3      월세   5000    60           19400.0      39.44         491.89
4      월세  40000   100           64000.0      93.71         682.96


In [10]:
# 극단값이 축을 다 잡아먹으므로 상하위 1%를 잘라낸다
lo, hi = df["JEONSE_PER_M2"].quantile([0.01, 0.99])
trimmed = df[(df["JEONSE_PER_M2"] >= lo) & (df["JEONSE_PER_M2"] <= hi)]
print(f"상하위 1% 제외: {len(df):,} → {len(trimmed):,}")

# 박스플롯은 원자료를 전부 브라우저로 보낸다. 39만 행이면 파일이 9MB를 넘어
# 열리지도 않으므로 그룹당 최대 1,500건을 무작위 추출한다(seed 고정).
# 사분위수 모양을 보는 목적이라 표본으로 충분하다.
box_src = trimmed[trimmed["BLDG_USG"].isin(["아파트", "연립다세대", "단독다가구"])]
# 전체를 섞은 뒤 그룹별로 앞에서 1,500건씩 — groupby.apply보다 열이 안 사라져 안전하다
box_src = box_src.sample(frac=1, random_state=0).groupby(["CGG_NM", "BLDG_USG"]).head(1500)
print(f"박스플롯용 표본: {len(box_src):,}건 (원본 {len(trimmed):,}건)")

fig = px.box(box_src,
             x="CGG_NM", y="JEONSE_PER_M2", color="BLDG_USG",
             labels={"CGG_NM": "자치구", "JEONSE_PER_M2": "㎡당 환산전세가 (만원)",
                     "BLDG_USG": "건물용도"})
fig.update_layout(title="자치구·건물용도별 ㎡당 환산전세가",
                  template="plotly_white", height=520, xaxis_tickangle=-45)
fig.write_html(HTML / "7_1.html", include_plotlyjs="cdn")
fig.show()

상하위 1% 제외: 416,037 → 407,715
박스플롯용 표본: 111,477건 (원본 407,715건)


In [11]:
# 계약일을 월 단위로 묶어 추세를 본다
t = trimmed.copy()
t["CTRT_DAY"] = pd.to_datetime(t["CTRT_DAY"], format="%Y%m%d", errors="coerce")
t = t[t["CTRT_DAY"].notna()]
t["month"] = t["CTRT_DAY"].dt.to_period("M").astype(str)

monthly = (t[t["BLDG_USG"] == "아파트"]
           .groupby(["month", "CGG_NM"], as_index=False)["JEONSE_PER_M2"]
           .mean().round(2))

# 거래가 많은 6개 구만 본다. 전부 그리면 선이 뒤엉켜 못 읽는다
top_gu = t[t["BLDG_USG"] == "아파트"]["CGG_NM"].value_counts().head(6).index

fig = px.line(monthly[monthly["CGG_NM"].isin(top_gu)],
              x="month", y="JEONSE_PER_M2", color="CGG_NM", markers=True,
              labels={"month": "계약월", "JEONSE_PER_M2": "㎡당 환산전세가 (만원)",
                      "CGG_NM": "자치구"})
fig.update_layout(title="아파트 ㎡당 환산전세가 월별 추이 (거래 상위 6개구)",
                  template="plotly_white", height=460)
fig.write_html(HTML / "7_2.html", include_plotlyjs="cdn")
fig.show()

print("저장된 HTML:", sorted(p.name for p in HTML.glob("*.html")))

저장된 HTML: ['13_1.html', '13_2.html', '1_1.html', '1_2.html', '1_3.html', '1_4.html', '2_3.html', '3_2.html', '4_2.html', '5_1.html', '6_1.html', '7_1.html', '7_2.html', '9_1.html', '9_2.html']


### 읽어낸 것

- 자치구 간 ㎡당 환산전세가 격차가 건물용도보다 크다. 어디냐가 무엇이냐보다 세다.
- 아파트는 분포 폭이 좁고 연립다세대·단독다가구는 넓다. 같은 구 안에서도 편차가 크다는 뜻이다.
- 월별 추이는 구마다 방향이 갈린다. 서울 전체를 하나의 시장으로 묶어 보면 이 차이가 지워진다.

주의: 전환율 5%는 가정이다. 실제 전환율은 시기·지역·주택유형에 따라 다르므로
이 값은 비교를 위한 공통 잣대일 뿐 실거래 전세가가 아니다.